# Notebook 01 — Collecte des données brutes
## Projet MSPR · Electio-Analytics · Pays de la Loire

---

### 🎯 Objectif de ce notebook

Ce notebook correspond à la **première étape du pipeline ETL** : la **collecte**.

Son rôle est de :
1. Charger les fichiers sources bruts (tels quels, sans modification)
2. Faire un **premier aperçu** de chaque fichier (dimensions, colonnes, types)
3. **Filtrer les données** sur la zone d'étude : région Pays de la Loire
4. **Extraire uniquement** les élections qui nous intéressent (1ers tours 2012/2017/2022)
5. Sauvegarder les extraits bruts dans `outputs/staging/` (zone `stg_raw`)

> ⚠️ **Règle importante** : Dans cette étape, on ne corrige **rien**.
> On ne modifie pas les valeurs, on ne nettoie pas les nulls.
> On copie fidèlement la donnée source pour garder une trace brute (principe de traçabilité).

---

### 📁 Fichiers sources utilisés

| Fichier | Contenu | Utilisation |
|---|---|---|
| `general_results.csv` | Participation électorale par bureau de vote | Variable cible : taux de participation |
| `candidats_results.csv` | Résultats par candidat et bureau de vote | Nuances politiques, volatilité, qui gagne |
| `crimes_delits_communes.csv` | Criminalité par commune et par année | Indicateur sécurité |
| `emploi_pop_active.CSV` | Population active et chômage par commune | Indicateur emploi/chômage |
| `base_cc_comparateur.csv` | Données socio-éco par commune (INSEE) | Pauvreté, population, entreprises |

---

### 👤 Réalisé par : Mickeal (Data Engineer)
### 📅 Étape : 1/4 — Collecte brute → `stg_raw`


---
## 0. Imports et configuration

In [2]:
# ── Imports standard ──────────────────────────────────────────────────────
import os
import pandas as pd
import numpy as np
from datetime import datetime

# Afficher toutes les colonnes dans les aperçus
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 120)
pd.set_option('display.float_format', '{:.2f}'.format)

print("✅ Imports OK")
print(f"   pandas  : {pd.__version__}")
print(f"   numpy   : {np.__version__}")


✅ Imports OK
   pandas  : 3.0.3
   numpy   : 2.4.4


In [3]:
# ── Constantes du projet ──────────────────────────────────────────────────
# Ces constantes sont partagées avec tous les notebooks du pipeline.
# Si tu changes la zone d'étude, tu n'as qu'à modifier ici.

PROJECT_NAME = "Electio-Analytics — MSPR EPSI 2026"
ZONE_ETUDE   = "Pays de la Loire"

# Codes des 5 départements de la région Pays de la Loire
# 44 = Loire-Atlantique, 49 = Maine-et-Loire, 53 = Mayenne
# 72 = Sarthe, 85 = Vendée
DEPTS_PDL = ['44', '49', '53', '72', '85']

# Identifiants des élections qu'on étudie (1ers tours uniquement)
# Format : AAAA_TYPE_tTOUR
ELECTIONS_CIBLES = [
    '2012_pres_t1',   # Présidentielle 2012, 1er tour
    '2017_pres_t1',   # Présidentielle 2017, 1er tour
    '2022_pres_t1',   # Présidentielle 2022, 1er tour
    '2012_legi_t1',   # Législatives 2012, 1er tour
    '2017_legi_t1',   # Législatives 2017, 1er tour
    '2022_legi_t1',   # Législatives 2022, 1er tour
]

# ── Chemins des fichiers sources (à adapter selon ton organisation) ────────
# Les fichiers sources sont placés dans data/raw_batches/
ROOT = ".."
PATH_RAW     = os.path.join(ROOT, "data", "raw_batches")
PATH_STG_RAW    = os.path.join(ROOT, "outputs", "staging", "raw")
PATH_STG_STD    = os.path.join(ROOT, "outputs", "staging", "std")
PATH_STG_REJECT = os.path.join(ROOT, "outputs", "staging", "reject")
PATH_STAGING    = os.path.join(ROOT, "outputs", "staging")  # dossier parent
PATH_OPS     = os.path.join(ROOT, "outputs", "ops")

# Créer les dossiers de sortie s'ils n'existent pas
os.makedirs(PATH_STG_RAW,    exist_ok=True)
os.makedirs(PATH_STG_STD,    exist_ok=True)
os.makedirs(PATH_STG_REJECT, exist_ok=True)
os.makedirs(PATH_OPS,        exist_ok=True)

# ── Chemins des fichiers sources ──────────────────────────────────────────
FICHIERS = {
    "general"   : os.path.join(PATH_RAW, "general_results.csv"),
    "candidats" : os.path.join(PATH_RAW, "candidats_results.csv"),
    "securite"  : os.path.join(PATH_RAW, "crimes_delits_communes.csv"),
    "emploi"    : os.path.join(PATH_RAW, "emploi_pop_active.CSV"),
    "socioeco"  : os.path.join(PATH_RAW, "base_cc_comparateur.csv"),
}

print("✅ Configuration OK")
print(f"   Zone étudiée : {ZONE_ETUDE}")
print(f"   Départements : {', '.join(DEPTS_PDL)}")
print(f"   Élections    : {ELECTIONS_CIBLES}")


✅ Configuration OK
   Zone étudiée : Pays de la Loire
   Départements : 44, 49, 53, 72, 85
   Élections    : ['2012_pres_t1', '2017_pres_t1', '2022_pres_t1', '2012_legi_t1', '2017_legi_t1', '2022_legi_t1']


---
## 1. Initialisation du batch de collecte

In [4]:
# ── Batch control : traçabilité de l'exécution ───────────────────────────
# On enregistre le début du batch pour pouvoir tracer l'exécution.
# Si le pipeline plante à mi-chemin, on saura exactement où.

BATCH_ID = f"B01_COLLECTE_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
BATCH_START = datetime.now()

print(f"🚀 Démarrage du batch : {BATCH_ID}")
print(f"   Heure de début     : {BATCH_START.strftime('%Y-%m-%d %H:%M:%S')}")


🚀 Démarrage du batch : B01_COLLECTE_20260523_210500
   Heure de début     : 2026-05-23 21:05:00


---
## 2. Fonctions utilitaires

In [5]:
# ── Fonction : aperçu rapide d'un DataFrame ──────────────────────────────
# Cette fonction est utilisée dans tous les notebooks pour avoir
# un résumé rapide d'un dataset : dimensions, types, nulls, aperçu.

def apercu(df, nom, n=3):
    """
    Affiche un résumé complet d'un DataFrame.
    
    Paramètres :
        df  : le DataFrame à analyser
        nom : nom du dataset (pour l'affichage)
        n   : nombre de lignes à afficher (défaut : 3)
    """
    print(f"\n{'='*60}")
    print(f"  Dataset : {nom}")
    print(f"{'='*60}")
    print(f"  Dimensions  : {df.shape[0]:,} lignes × {df.shape[1]} colonnes")
    print(f"  Mémoire     : {df.memory_usage(deep=True).sum() / 1024**2:.1f} MB")
    print()
    
    # Résumé des colonnes : type + nb nulls + % nulls
    resume = pd.DataFrame({
        'type'     : df.dtypes,
        'nb_nulls' : df.isnull().sum(),
        'pct_nulls': (df.isnull().sum() / len(df) * 100).round(2)
    })
    print("  Colonnes :")
    print(resume.to_string())
    print()
    
    # Aperçu des premières lignes
    print(f"  Aperçu ({n} premières lignes) :")
    print(df.head(n).to_string())
    print()

print("✅ Fonctions utilitaires définies")


✅ Fonctions utilitaires définies


---
## 3. Collecte — `general_results.csv`
### Participation électorale (inscrits, votants, abstentions)

Ce fichier contient **tous les résultats de participation** pour toutes les élections françaises
depuis 1999, au niveau bureau de vote.

On va :
1. Lire le fichier en chunks (il est volumineux)
2. Filtrer sur nos 6 élections cibles + les 5 départements PDL
3. Sauvegarder l'extrait brut → `stg_raw_general.csv`


In [6]:
# ── Lecture en chunks pour gérer la taille du fichier ────────────────────
# Le fichier est volumineux, on le lit par morceaux (chunks) de 100 000 lignes
# pour éviter de saturer la mémoire RAM.

print(f"📂 Lecture de : general_results.csv")
print(f"   Filtre élections : {ELECTIONS_CIBLES}")
print(f"   Filtre départements PDL : {DEPTS_PDL}")
print()

morceaux = []   # liste pour stocker chaque chunk filtré
total_lu = 0    # compteur total de lignes lues

for chunk in pd.read_csv(
    FICHIERS["general"],
    sep=None,            # détection automatique du séparateur
    engine='python',
    chunksize=100_000,   # lecture par blocs de 100k lignes
    dtype={'code_departement': str}  # forcer string pour éviter 44 → 44.0
):
    total_lu += len(chunk)
    
    # Normaliser le code département : "1" → "01", "44" → "44"
    chunk['code_departement'] = chunk['code_departement'].str.zfill(2)
    
    # Filtre double : élection cible ET département PDL
    masque = (
        chunk['id_election'].isin(ELECTIONS_CIBLES) &
        chunk['code_departement'].isin(DEPTS_PDL)
    )
    
    if masque.any():
        morceaux.append(chunk[masque])

# Assembler tous les morceaux filtrés en un seul DataFrame
df_general_raw = pd.concat(morceaux, ignore_index=True)

print(f"✅ Lecture terminée")
print(f"   Lignes lues au total   : {total_lu:,}")
print(f"   Lignes après filtrage  : {len(df_general_raw):,}")
print(f"   Taux de sélection      : {len(df_general_raw)/total_lu*100:.2f}%")


📂 Lecture de : general_results.csv
   Filtre élections : ['2012_pres_t1', '2017_pres_t1', '2022_pres_t1', '2012_legi_t1', '2017_legi_t1', '2022_legi_t1']
   Filtre départements PDL : ['44', '49', '53', '72', '85']

✅ Lecture terminée
   Lignes lues au total   : 3,162,440
   Lignes après filtrage  : 19,824
   Taux de sélection      : 0.63%


In [7]:
# ── Aperçu du dataset général filtré ─────────────────────────────────────
apercu(df_general_raw, "general_results — PDL filtré")

# Vérification : combien de lignes par élection ?
print("  Répartition par élection :")
print(df_general_raw['id_election'].value_counts().sort_index().to_string())
print()
print("  Répartition par département :")
print(df_general_raw['code_departement'].value_counts().sort_index().to_string())



  Dataset : general_results — PDL filtré
  Dimensions  : 19,824 lignes × 25 colonnes
  Mémoire     : 6.5 MB

  Colonnes :
                               type  nb_nulls  pct_nulls
id_election                     str         0       0.00
id_brut_miom                    str         0       0.00
code_departement                str         0       0.00
libelle_departement             str         0       0.00
code_canton                 float64     13394      67.56
libelle_canton               object     19824     100.00
code_commune                    str         0       0.00
libelle_commune                 str         0       0.00
code_circonscription        float64         0       0.00
libelle_circonscription      object      6430      32.44
code_bv                         str         0       0.00
inscrits                      int64         0       0.00
abstentions                   int64         0       0.00
votants                       int64         0       0.00
blancs                

In [8]:
# ── Sauvegarde stg_raw_general.csv ───────────────────────────────────────
# On sauvegarde l'extrait BRUT — aucune modification des valeurs.
# C'est notre "boîte noire" : si quelque chose plante plus tard,
# on peut toujours revenir à cette version originale.

chemin_sortie = os.path.join(PATH_STG_RAW, "stg_raw_general.csv")
df_general_raw.to_csv(chemin_sortie, index=False, encoding='utf-8')

print(f"💾 Sauvegardé : stg_raw_general.csv")
print(f"   Chemin  : {chemin_sortie}")
print(f"   Lignes  : {len(df_general_raw):,}")
print(f"   Taille  : {os.path.getsize(chemin_sortie)/1024:.1f} KB")


💾 Sauvegardé : stg_raw_general.csv
   Chemin  : ..\outputs\staging\raw\stg_raw_general.csv
   Lignes  : 19,824
   Taille  : 2893.0 KB


---
## 4. Collecte — `candidats_results.csv`
### Résultats par candidat (nuances politiques, voix, volatilité)

Ce fichier contient les résultats **par candidat** pour chaque bureau de vote.
Il nous permettra de calculer :
- **Qui gagne** dans les Pays de la Loire (quel candidat / quelle nuance)
- La **volatilité politique** entre 2012, 2017 et 2022

### Référentiel des nuances politiques
Les nuances sont des codes qui représentent la tendance politique du candidat.
Voici celles présentes dans nos données :

| Nuance | Signification |
|---|---|
| HOLL, SOC, DVG | Gauche (PS, divers gauche) |
| MELE, FI, EXG | Gauche radicale (LFI, extrême gauche) |
| SARK, UMP, LR, DVD | Droite (UMP/LR, divers droite) |
| LEPE, FN, RN | Extrême droite (FN/RN) |
| BAYR, MACR, REM, ENS | Centre (Bayrou, Macron, LREM/Renaissance) |
| ECO, JOLY | Écologie |


In [9]:
# ── Lecture et filtrage candidats ────────────────────────────────────────
print(f"📂 Lecture de : candidats_results.csv")
print(f"   Filtre élections : {ELECTIONS_CIBLES}")
print(f"   Filtre départements PDL : {DEPTS_PDL}")
print()

morceaux_cand = []
total_lu_cand = 0

for chunk in pd.read_csv(
    FICHIERS["candidats"],
    sep=None,
    engine='python',
    chunksize=100_000,
    dtype={'code_departement': str}
):
    total_lu_cand += len(chunk)
    
    # Normaliser le code département
    chunk['code_departement'] = chunk['code_departement'].astype(str).str.zfill(2)
    
    # Filtre : élections cibles + départements PDL
    masque = (
        chunk['id_election'].isin(ELECTIONS_CIBLES) &
        chunk['code_departement'].isin(DEPTS_PDL)
    )
    
    if masque.any():
        morceaux_cand.append(chunk[masque])

df_candidats_raw = pd.concat(morceaux_cand, ignore_index=True)

print(f"✅ Lecture terminée")
print(f"   Lignes lues au total   : {total_lu_cand:,}")
print(f"   Lignes après filtrage  : {len(df_candidats_raw):,}")


📂 Lecture de : candidats_results.csv
   Filtre élections : ['2012_pres_t1', '2017_pres_t1', '2022_pres_t1', '2012_legi_t1', '2017_legi_t1', '2022_legi_t1']
   Filtre départements PDL : ['44', '49', '53', '72', '85']

✅ Lecture terminée
   Lignes lues au total   : 27,524,743
   Lignes après filtrage  : 222,280


In [10]:
# ── Aperçu candidats ─────────────────────────────────────────────────────
apercu(df_candidats_raw, "candidats_results — PDL filtré")

# Vérification : nuances disponibles par élection présidentielle
print("  Nuances présidentielles disponibles (PDL) :")
pres = df_candidats_raw[df_candidats_raw['id_election'].str.contains('pres')]
print(pres.groupby(['id_election', 'nuance']).size().reset_index(name='nb_bv').to_string(index=False))



  Dataset : candidats_results — PDL filtré
  Dimensions  : 222,280 lignes × 18 colonnes
  Mémoire     : 92.7 MB

  Colonnes :
                         type  nb_nulls  pct_nulls
id_election               str         0       0.00
id_brut_miom              str         0       0.00
code_departement          str         0       0.00
code_commune           object         0       0.00
code_bv                object         0       0.00
no_panneau            float64         0       0.00
voix                    int64         0       0.00
ratio_voix_inscrits   float64         0       0.00
ratio_voix_exprimes   float64        24       0.01
nuance                 object     77050      34.66
sexe                   object     66366      29.86
nom                       str         0       0.00
prenom                    str         0       0.00
liste                  object    222280     100.00
libelle_abrege_liste   object    222280     100.00
libelle_etendu_liste   object    222280     100.00
nom_te

In [11]:
# ── Sauvegarde stg_raw_candidats.csv ─────────────────────────────────────
chemin_sortie = os.path.join(PATH_STG_RAW, "stg_raw_candidats.csv")
df_candidats_raw.to_csv(chemin_sortie, index=False, encoding='utf-8')

print(f"💾 Sauvegardé : stg_raw_candidats.csv")
print(f"   Chemin  : {chemin_sortie}")
print(f"   Lignes  : {len(df_candidats_raw):,}")
print(f"   Taille  : {os.path.getsize(chemin_sortie)/1024:.1f} KB")


💾 Sauvegardé : stg_raw_candidats.csv
   Chemin  : ..\outputs\staging\raw\stg_raw_candidats.csv
   Lignes  : 222,280
   Taille  : 17884.2 KB


---
## 5. Collecte — `crimes_delits_communes.csv`
### Indicateur sécurité / criminalité

Ce fichier contient les données de criminalité par commune et par année.
Il est structuré en **format long** : une ligne = 1 commune × 1 année × 1 type d'indicateur.

On filtre sur les années proches de nos élections : 2011-2012, 2016-2017, 2021-2022.


In [12]:
# ── Lecture sécurité ─────────────────────────────────────────────────────
print(f"📂 Lecture de : crimes_delits_communes.csv")

# On lit un premier aperçu pour voir la structure
df_secu_apercu = pd.read_csv(
    FICHIERS["securite"],
    sep=None, engine='python',
    nrows=5
)
print("  Colonnes disponibles :")
print(list(df_secu_apercu.columns))
print()
print("  Exemple :")
print(df_secu_apercu.head(3).to_string())


📂 Lecture de : crimes_delits_communes.csv
  Colonnes disponibles :
['CODGEO_2025', 'annee', 'indicateur', 'unite_de_compte', 'nombre', 'taux_pour_mille', 'est_diffuse', 'insee_pop', 'insee_pop_millesime', 'insee_log', 'insee_log_millesime', 'complement_info_nombre', 'complement_info_taux']

  Exemple :
   CODGEO_2025  annee                               indicateur unite_de_compte  nombre taux_pour_mille est_diffuse  insee_pop  insee_pop_millesime  insee_log  insee_log_millesime complement_info_nombre complement_info_taux
0         1001   2016      Violences physiques intrafamiliales         Victime    0.00       0,0000000        diff        767                 2016        348                 2016                    NaN                  NaN
1         1001   2016  Violences physiques hors cadre familial         Victime     NaN             NaN       ndiff        767                 2016        348                 2016              1,3620690            0,9598386
2         1001   2016      

In [13]:
# ── Filtrage sécurité sur PDL et années proches des élections ────────────
# Le code commune commence par le code département
# Ex : commune 44001 → département 44 (Loire-Atlantique)
# On filtre sur les années 2011, 2012, 2016, 2017, 2021, 2022

ANNEES_SECU = [2011, 2012, 2016, 2017, 2021, 2022]

df_secu_raw = pd.read_csv(
    FICHIERS["securite"],
    sep=None, engine='python',
    dtype={'CODGEO_2025': str}
)

# Extraire le code département depuis le code commune (2 premiers chiffres)
df_secu_raw['code_departement'] = df_secu_raw['CODGEO_2025'].str[:2]

# Filtre : département PDL + années cibles
masque_secu = (
    df_secu_raw['code_departement'].isin(DEPTS_PDL) &
    df_secu_raw['annee'].isin(ANNEES_SECU)
)
df_secu_raw = df_secu_raw[masque_secu].copy()

print(f"✅ Lecture terminée")
print(f"   Lignes après filtrage : {len(df_secu_raw):,}")
print()

apercu(df_secu_raw, "crimes_delits — PDL filtré")

# Types d'indicateurs disponibles
print("  Types d'indicateurs sécurité disponibles :")
print(df_secu_raw['indicateur'].value_counts().to_string())


✅ Lecture terminée
   Lignes après filtrage : 73,680


  Dataset : crimes_delits — PDL filtré
  Dimensions  : 73,680 lignes × 14 colonnes
  Mémoire     : 13.0 MB

  Colonnes :
                           type  nb_nulls  pct_nulls
CODGEO_2025                 str         0       0.00
annee                     int64         0       0.00
indicateur                  str         0       0.00
unite_de_compte             str         0       0.00
nombre                  float64     39599      53.74
taux_pour_mille             str     39599      53.74
est_diffuse                 str         0       0.00
insee_pop                 int64         0       0.00
insee_pop_millesime       int64         0       0.00
insee_log                 int64         0       0.00
insee_log_millesime       int64         0       0.00
complement_info_nombre      str     34081      46.26
complement_info_taux        str     34081      46.26
code_departement            str         0       0.00

  Aperçu (3 premières lignes

In [14]:
# ── Sauvegarde stg_raw_securite.csv ──────────────────────────────────────
chemin_sortie = os.path.join(PATH_STG_RAW, "stg_raw_securite.csv")
df_secu_raw.to_csv(chemin_sortie, index=False, encoding='utf-8')

print(f"💾 Sauvegardé : stg_raw_securite.csv")
print(f"   Chemin  : {chemin_sortie}")
print(f"   Lignes  : {len(df_secu_raw):,}")
print(f"   Taille  : {os.path.getsize(chemin_sortie)/1024:.1f} KB")


💾 Sauvegardé : stg_raw_securite.csv
   Chemin  : ..\outputs\staging\raw\stg_raw_securite.csv
   Lignes  : 73,680
   Taille  : 7337.9 KB


---
## 6. Collecte — `emploi_pop_active.CSV`
### Indicateur emploi / chômage

Ce fichier contient les données d'emploi et chômage par commune.
Il est structuré en **format large** : une ligne = 1 commune, avec les données
pour plusieurs années (millésimes des recensements INSEE).

Correspondance des millésimes avec nos élections :
- `P10_*` → Recensement 2010 → proxy pour l'élection **2012**
- `P15_*` → Recensement 2015 → proxy pour l'élection **2017**
- `P21_*` → Recensement 2021 → proxy pour l'élection **2022**


In [15]:
# ── Lecture emploi ───────────────────────────────────────────────────────
print(f"📂 Lecture de : emploi_pop_active.CSV")

df_emploi_raw = pd.read_csv(
    FICHIERS["emploi"],
    sep=None, engine='python',
    dtype={'CODGEO': str}
)

# Extraire le code département depuis le code commune (2 premiers chiffres)
df_emploi_raw['code_departement'] = df_emploi_raw['CODGEO'].str[:2]

# Filtre : département PDL uniquement
df_emploi_raw = df_emploi_raw[
    df_emploi_raw['code_departement'].isin(DEPTS_PDL)
].copy()

print(f"✅ Lecture terminée")
print(f"   Lignes après filtrage : {len(df_emploi_raw):,}")
print()

# On sélectionne uniquement les colonnes utiles pour éviter un fichier trop lourd
# CODGEO = code commune, puis les colonnes chômage et population des 3 millésimes
COLS_EMPLOI_UTILES = [
    'CODGEO', 'code_departement',
    # Millésime 2010 (proxy 2012)
    'P10_POP1564',    # Population active 15-64 ans
    'P10_CHOM1564',   # Chômeurs 15-64 ans
    # Millésime 2015 (proxy 2017)
    'P15_POP1564',
    'P15_CHOM1564',
    # Millésime 2021 (proxy 2022)
    'P21_POP1564',
    'P21_CHOM1564',
]

# Garder seulement les colonnes qui existent dans le fichier
cols_presentes = [c for c in COLS_EMPLOI_UTILES if c in df_emploi_raw.columns]
df_emploi_raw = df_emploi_raw[cols_presentes].copy()

apercu(df_emploi_raw, "emploi_pop_active — PDL filtré (colonnes sélectionnées)")


📂 Lecture de : emploi_pop_active.CSV
✅ Lecture terminée
   Lignes après filtrage : 1,232


  Dataset : emploi_pop_active — PDL filtré (colonnes sélectionnées)
  Dimensions  : 1,232 lignes × 8 colonnes
  Mémoire     : 0.1 MB

  Colonnes :
                     type  nb_nulls  pct_nulls
CODGEO                str         0       0.00
code_departement      str         0       0.00
P10_POP1564       float64         2       0.16
P10_CHOM1564      float64         2       0.16
P15_POP1564       float64         2       0.16
P15_CHOM1564      float64         2       0.16
P21_POP1564       float64         2       0.16
P21_CHOM1564      float64         2       0.16

  Aperçu (3 premières lignes) :
      CODGEO code_departement  P10_POP1564  P10_CHOM1564  P15_POP1564  P15_CHOM1564  P21_POP1564  P21_CHOM1564
16211  44001               44      1151.00         99.00      1200.00         84.00      1256.43         70.75
16212  44002               44      1992.00         93.00      2278.00        130.00 

C:\Users\OMEN\AppData\Local\Temp\ipykernel_26944\2715050763.py:11: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_emploi_raw['code_departement'] = df_emploi_raw['CODGEO'].str[:2]


In [16]:
# ── Sauvegarde stg_raw_emploi.csv ────────────────────────────────────────
chemin_sortie = os.path.join(PATH_STG_RAW, "stg_raw_emploi.csv")
df_emploi_raw.to_csv(chemin_sortie, index=False, encoding='utf-8')

print(f"💾 Sauvegardé : stg_raw_emploi.csv")
print(f"   Chemin  : {chemin_sortie}")
print(f"   Lignes  : {len(df_emploi_raw):,}")
print(f"   Taille  : {os.path.getsize(chemin_sortie)/1024:.1f} KB")


💾 Sauvegardé : stg_raw_emploi.csv
   Chemin  : ..\outputs\staging\raw\stg_raw_emploi.csv
   Lignes  : 1,232
   Taille  : 122.4 KB


---
## 7. Collecte — `base_cc_comparateur.csv`
### Indicateurs socio-économiques (pauvreté, population, entreprises)

Ce fichier INSEE contient par commune les données de :
- **Population** : `P22_POP` (2022), `P16_POP` (2016)
- **Pauvreté** : `PR_MD60_23` (taux pauvreté 2023)
- **Entreprises** : `ETTOT24` (nb entreprises 2024)
- **Médiane salaire** : `MED_SL23`
- **Chômage** (complément) : `P22_CHOM1564`

> Note : Ce fichier ne contient que les données **récentes** (2016-2022).
> Pour avoir des indicateurs historiques 2012, on utilise `emploi_pop_active.CSV`.


In [17]:
# ── Lecture socio-éco ─────────────────────────────────────────────────────
print(f"📂 Lecture de : base_cc_comparateur.csv")

df_socio_raw = pd.read_csv(
    FICHIERS["socioeco"],
    sep=None, engine='python',
    dtype={'CODGEO': str}
)

# Extraire le code département
df_socio_raw['code_departement'] = df_socio_raw['CODGEO'].str[:2]

# Filtre : département PDL
df_socio_raw = df_socio_raw[
    df_socio_raw['code_departement'].isin(DEPTS_PDL)
].copy()

print(f"✅ Lecture terminée")
print(f"   Lignes après filtrage : {len(df_socio_raw):,}")
print()

# Colonnes utiles pour notre projet
COLS_SOCIO_UTILES = [
    'CODGEO', 'code_departement',
    'P22_POP',        # Population 2022
    'P16_POP',        # Population 2016
    'P22_MEN',        # Nombre de ménages 2022
    'MED_SL23',       # Médiane salaire 2023
    'PR_MD60_23',     # Taux de pauvreté 2023
    'P22_CHOM1564',   # Chômeurs 2022
    'P22_ACT1564',    # Population active 2022
    'P22_EMPLT',      # Emplois 2022
    'ETTOT24',        # Nb total d'entreprises 2024
]

cols_presentes = [c for c in COLS_SOCIO_UTILES if c in df_socio_raw.columns]
df_socio_raw = df_socio_raw[cols_presentes].copy()

apercu(df_socio_raw, "base_cc_comparateur — PDL filtré (colonnes sélectionnées)")


📂 Lecture de : base_cc_comparateur.csv
✅ Lecture terminée
   Lignes après filtrage : 1,233


  Dataset : base_cc_comparateur — PDL filtré (colonnes sélectionnées)
  Dimensions  : 1,233 lignes × 11 colonnes
  Mémoire     : 0.1 MB

  Colonnes :
                     type  nb_nulls  pct_nulls
CODGEO                str         0       0.00
code_departement      str         0       0.00
P22_POP           float64         5       0.41
P16_POP           float64         5       0.41
P22_MEN           float64         5       0.41
MED_SL23              str         5       0.41
PR_MD60_23            str         5       0.41
P22_CHOM1564      float64         5       0.41
P22_ACT1564       float64         5       0.41
P22_EMPLT         float64         5       0.41
ETTOT24           float64         5       0.41

  Aperçu (3 premières lignes) :
      CODGEO code_departement  P22_POP  P16_POP  P22_MEN MED_SL23 PR_MD60_23  P22_CHOM1564  P22_ACT1564  P22_EMPLT  ETTOT24
16201  44001               44  2052.

In [18]:
# ── Sauvegarde stg_raw_socioeco.csv ──────────────────────────────────────
chemin_sortie = os.path.join(PATH_STG_RAW, "stg_raw_socioeco.csv")
df_socio_raw.to_csv(chemin_sortie, index=False, encoding='utf-8')

print(f"💾 Sauvegardé : stg_raw_socioeco.csv")
print(f"   Chemin  : {chemin_sortie}")
print(f"   Lignes  : {len(df_socio_raw):,}")
print(f"   Taille  : {os.path.getsize(chemin_sortie)/1024:.1f} KB")


💾 Sauvegardé : stg_raw_socioeco.csv
   Chemin  : ..\outputs\staging\raw\stg_raw_socioeco.csv
   Lignes  : 1,233
   Taille  : 116.4 KB


---
## 8. Bilan de la collecte — Récapitulatif `stg_raw`


In [19]:
# ── Bilan final de la collecte ────────────────────────────────────────────
# On liste tous les fichiers stg_raw produits avec leurs caractéristiques

print("=" * 65)
print("  BILAN COLLECTE — ZONE stg_raw")
print("=" * 65)
print()

fichiers_produits = {
    "stg_raw_general.csv"  : df_general_raw,
    "stg_raw_candidats.csv": df_candidats_raw,
    "stg_raw_securite.csv" : df_secu_raw,
    "stg_raw_emploi.csv"   : df_emploi_raw,
    "stg_raw_socioeco.csv" : df_socio_raw,
}

bilan = []
for nom, df in fichiers_produits.items():
    chemin = os.path.join(PATH_STAGING, nom)
    taille = os.path.getsize(chemin) / 1024 if os.path.exists(chemin) else 0
    bilan.append({
        'fichier'    : nom,
        'lignes'     : len(df),
        'colonnes'   : df.shape[1],
        'nulls_total': int(df.isnull().sum().sum()),
        'taille_kb'  : round(taille, 1)
    })

df_bilan = pd.DataFrame(bilan)
print(df_bilan.to_string(index=False))
print()

# Calcul du taux global de nulls
total_cellules = sum(d.shape[0] * d.shape[1] for d in fichiers_produits.values())
total_nulls    = sum(int(d.isnull().sum().sum()) for d in fichiers_produits.values())
print(f"  Taux global de nulls : {total_nulls/total_cellules*100:.2f}%")
print()

# Durée d'exécution du batch
duree = (datetime.now() - BATCH_START).seconds
print(f"  ⏱  Durée d'exécution : {duree} secondes")
print(f"  ✅ Batch {BATCH_ID} terminé avec succès")


  BILAN COLLECTE — ZONE stg_raw

              fichier  lignes  colonnes  nulls_total  taille_kb
  stg_raw_general.csv   19824        25        58941          0
stg_raw_candidats.csv  222280        18      1254840          0
 stg_raw_securite.csv   73680        14       147360          0
   stg_raw_emploi.csv    1232         8           12          0
 stg_raw_socioeco.csv    1233        11           45          0

  Taux global de nulls : 26.32%

  ⏱  Durée d'exécution : 319 secondes
  ✅ Batch B01_COLLECTE_20260523_210500 terminé avec succès


In [20]:
# ── Enregistrement du batch dans ops_batch_control.csv ───────────────────
# On trace l'exécution de ce notebook dans le fichier de monitoring.

batch_file = os.path.join(PATH_OPS, "ops_batch_control.csv")

batch_row = pd.DataFrame([{
    "batch_id"      : BATCH_ID,
    "notebook"      : "01_collecte",
    "statut"        : "SUCCESS",
    "zone_etude"    : ZONE_ETUDE,
    "nb_fichiers"   : len(fichiers_produits),
    "nb_lignes_total": sum(len(d) for d in fichiers_produits.values()),
    "duree_sec"     : (datetime.now() - BATCH_START).seconds,
    "run_ts"        : datetime.now().isoformat(),
    "note"          : "Collecte brute OK — 5 fichiers stg_raw produits"
}])

if os.path.exists(batch_file):
    df_existing = pd.read_csv(batch_file)
    batch_row   = pd.concat([df_existing, batch_row], ignore_index=True)

batch_row.to_csv(batch_file, index=False)
print(f"✅ Batch enregistré dans ops_batch_control.csv")


✅ Batch enregistré dans ops_batch_control.csv


---
## ✅ Récapitulatif — Ce qu'on a produit

| Fichier produit | Contenu | Étape suivante |
|---|---|---|
| `stg_raw_general.csv` | Participation PDL, 6 élections | → Notebook 02 : nettoyage |
| `stg_raw_candidats.csv` | Candidats PDL, 6 élections | → Notebook 02 : nettoyage |
| `stg_raw_securite.csv` | Criminalité PDL, 2011-2022 | → Notebook 02 : nettoyage |
| `stg_raw_emploi.csv` | Chômage PDL, 3 millésimes | → Notebook 02 : nettoyage |
| `stg_raw_socioeco.csv` | Socio-éco PDL, INSEE | → Notebook 02 : nettoyage |

---
> **Suite : Notebook 02 — Staging & Qualité**
> Nettoyage, typage, standardisation, détection des rejets → `stg_std` + `stg_reject`
